# Europeana APIs: retrieving full text, step by step

### What is Europeana ?

[Europeana](https://www.europeana.eu/) aggregates digitized cultural heritage
from thousands of libraries, museums, and archives across Europe (including
Gallica).

Where Gallica is a single institution's digital library with its own bespoke
API, Europeana is an **aggregator** : it doesn't host most of the original
files, but it harvests, standardizes, and re-exposes metadata from hundreds of
providers using a shared data model. Practically, this means:

- Europeana's API responses are **consistent** across very different
  kinds of objects.
- Europeana **doesn't always hold the full text** itself : sometimes you'll
  be redirected back to the original provider's site or API to get it.
- Because Europeana is a *suite* of APIs rather than a single endpoint, the
  same kind of information (e.g. "the text on this page") can be reached two
  different ways: through dedicated full-text endpoints, or through IIIF.

### The Europeana API Suite 

**Starting points:**

- **[Swagger console](https://api.europeana.eu/console/index.html?url=docs/v3/iiif.json#/)** : an interactive page where you can test calls directly in your browser.
- **[API key](https://pro.europeana.eu/pages/get-api)** : unlike Gallica, most Europeana endpoints require a free API key, passed as the `wskey` parameter in every request. You can request a Personal API Key by creating a Europeana account for free. Once you have a key, you can save it as the default API key for Europeana in your **[browser](https://europeana.atlassian.net/wiki/spaces/EF/pages/2358673409/How+to+set+up+your+browser+to+access+the+Europeana+APIs)**.

Europeana does not expose full text (OCR / transcriptions) through a single call. Getting from *"I have a search term"* to *"I have the transcribed text of a page"* means chaining **three different APIs**, each responsible for one part of the job:

| Step | API | What it gives you |
|---|---|---|
| 1 | **Search API** | Which *records* match your query (dataset ID + record ID) |
| 2 | **Annotation API** (`annopage` endpoint) | Which *pages* of a record have text, and the IIIF image regions they correspond to |
| 3 | **Fulltext API** | The actual OCR / transcribed *text* of a page |

This notebook walks through the three steps using a concrete use case → **find occurrences of the word "révolution" in French-language resources from the 18th century that actually contain full text, then retrieve a selection of that text.**

> Note: Europeana's public documentation for these endpoints is sometimes incomplete or evolves between versions.
> 
> A word on IIIF: **IIIF** (International Image Interoperability Framework) is a set of open, shared standards that let cultural heritage institutions publish images (and, as an extension, text, audio and video) so that any compatible viewer or tool can display, zoom into, or query them in a consistent way : see https://iiif.io/ for the full specification.

## Setup

### Dependendies
You only need the `httpx` and `pandas` libraries. 

### Personal API key
Get a free API key at https://pro.europeana.eu/pages/get-api and paste it below. 
> Note that the Search API requires a key, while the IIIF and Fulltext APIs do not.
> **Do not share your personal API key.**


In [134]:
import httpx
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

europeana_client_access_token = os.getenv("WSKEY") 

## Step 1 : Using Search API to find records

We're searching for **"révolution"**, restricted to:
- French-language resources → `qf=LANGUAGE:fr`
- the 18th century → `qf=YEAR:[1700 TO 1799]`
- records that actually contain full text → `qf=TEXT_FULLTEXT:true`

`text_fulltext` and `year` are both standard Search API filter (`qf`) fields, so this call uses the general-purpose Search API endpoint (`/record/v2/search.json`) rather than the more narrowly scoped fulltext-search endpoint. This way, we can combine language, date and full-text filters in one request.

Each result's `id` field is the **Europeana ID**, formatted as `/datasetID/recordID`. That single field gives us both the dataset ID and the record ID we need for Step 2.


In [116]:
# Initiate the query
response = httpx.get(
    "https://api.europeana.eu/fulltext/search.json",
    params={
        "wskey": WSKEY,
        "query": "révolution",
        "qf": ["LANGUAGE:fr", "proxy_dcterms_issued:[1700 TO 1799]", "is_fulltext:true"],
        "rows": 20,
        "profile": "minimal",
    },
)

response.raise_for_status()
data = response.json()

# Always inspect the raw response first this tells you exactly which keys are available
print("Top-level keys:", list(data.keys()))
print("Total results:", data.get("totalResults"))


Top-level keys: ['apikey', 'success', 'requestNumber', 'itemsCount', 'totalResults', 'items']
Total results: 843


In [120]:
# Extract Europeana ID, infer dataset ID and record ID, and extract title
records = []
for item in data.get("items", []):
    europeana_id = item["id"]
    title = item["title"]
    dataset_id, record_id = europeana_id.strip("/").split("/", 1)
    records.append({
        "europeana_id": europeana_id,
        "dataset_id": dataset_id,
        "record_id": record_id,
        "title": title,
    })

# Create a dataframe to preview results
records_df = pd.DataFrame(records)
records_df

,europeana_id,dataset_id,record_id,title
0,/9200396/BibliographicResource_3000118436071,9200396,BibliographicResource_3000118436071,[Journal historique et littéraire - 1792-06-15]
1,/9200396/BibliographicResource_3000118435487,9200396,BibliographicResource_3000118435487,[La clef du cabinet des princes de l'Europe - 1742-02-01]
2,/9200396/BibliographicResource_3000118436070,9200396,BibliographicResource_3000118436070,[Journal historique et littéraire - 1792-12-15]
3,/9200396/BibliographicResource_3000118435453,9200396,BibliographicResource_3000118435453,[La clef du cabinet des princes de l'Europe - 1762-09-01]
4,/9200396/BibliographicResource_3000118436308,9200396,BibliographicResource_3000118436308,[Journal historique et littéraire - 1794-04-01]
5,/9200396/BibliographicResource_3000118436082,9200396,BibliographicResource_3000118436082,[Journal historique et littéraire - 1792-09-01]
6,/9200396/BibliographicResource_3000118436017,9200396,BibliographicResource_3000118436017,[Journal historique et littéraire - 1790-02-15]
7,/9200396/BibliographicResource_3000118435870,9200396,BibliographicResource_3000118435870,[Journal historique et littéraire - 1793-07-15]
8,/9200396/BibliographicResource_3000118435879,9200396,BibliographicResource_3000118435879,[Journal historique et littéraire - 1793-04-15]
9,/9200396/BibliographicResource_3000118435563,9200396,BibliographicResource_3000118435563,[La clef du cabinet des princes de l'Europe - 1772-10-01]


## Step 2 : Annotation API (`annopage`) to find pages

For a given record, full text lives on one or more **pages**, each identified by a page number. The `annopage` endpoint returns, for a given page, the list of text annotations *and* the IIIF image regions each annotation corresponds to:

```
https://iiif.europeana.eu/presentation/{RECORD_ID}/annopage/{PAGE_ID}?format=3
```

- `DATASET_ID` and `RECORD_ID` are the `/dataset_ID/record_ID` path from Step 1.
- `PAGE_ID` is a page ID.
- `format=3` asks for IIIF Presentation API v3 syntax (`body` / `target`), which is what current Europeana manifests use; the default (`format=2`) uses the older `resource` / `on` syntax (from the IIIF Presentation API v3).

> Note : our example resource for this notebook will be the first result of our Search response : Journal historique et littéraire, 1791-12-15.

In [128]:
# Retrieve pageIDs for our example resource
def get_page_ids(dataset_id: str, record_id: str, lang: str | None = None) -> list[str]:

    url = (
        f"https://api.europeana.eu/presentation/"
        f"{dataset_id}/{record_id}/annopage"
    )

    params = {}
    if lang is not None:
        params["lang"] = lang

    response = httpx.get(url, params=params)
    response.raise_for_status()

    data = response.json()

    return [
        item["id"].split("?")[0].rsplit("/", 1)[-1]
        for item in data.get("items", [])
    ]


# Example, here we're using the Journal historique et littéraire (europeanaID : /9200396/BibliographicResource_3000118436071)
example = records_df.iloc[0]

page_ids = get_page_ids(
    example["dataset_id"],
    example["record_id"],
    lang="fr",
)

print(f"Number of pageIDs available for {title}: {len(page_ids)}")
print(f"List: {page_ids}")

Number of pageIDs available for ['Journal historique et littéraire - 1791-12-15'] : 79
List : ['02467f5', '03d9db1', '0d430be', '1420d62', '17bbe1c', '1903d15', '1942609', '1a2973b', '2c3ba00', '2ee0cf8', '30d20c6', '311b325', '36e06d2', '3c7706b', '3df2601', '3ef43fa', '47d73e4', '4c6d06a', '4db1b2e', '4e913c3', '5898ce4', '58af913', '61d5289', '63b58f5', '63e6071', '6704b59', '6c31052', '6ccfedc', '6d638ab', '6e16d0c', '71342ca', '740ddf8', '75fc67c', '7865ab5', '79ee482', '7c013d4', '7eaf00f', '7fdf0b7', '81453c2', '81a2d2c', '8b08541', '8b422a8', '8d2d09b', '8d7af4d', '8e9d71c', '966cb05', '9d75630', '9dabb3e', '9fd1c62', 'a26ef27', 'a5cae08', 'a9bf6ec', 'b01a5ec', 'b156de7', 'b3a263b', 'b5193a0', 'b56d0bf', 'b7e8ba3', 'b7ed55c', 'b882730', 'b8bcab5', 'bdcb338', 'c34252a', 'c4eef4c', 'c64ca6e', 'c8d1ede', 'cadb391', 'ccce373', 'cfa5c8c', 'd6e9b15', 'da9690f', 'e1b6f7e', 'e9d8df3', 'f1b6004', 'f2a749c', 'f2f2350', 'f3b9906', 'fcba2d1', 'fe7458f']


## Step 3 : Retrieve the IIIF image and OCR text for each page

Now that we have discovered the Annotation pageIDs for a record, we can retrieve the content associated with each page.

To retrieve the IIIF image, we must go through the **IIIF Manifest canvas list**, where each canvas corresponds to a page image.

### Retrieve the IIIF image for a page

Each Annotation Page references a canvas in the IIIF Presentation API.

From this canvas we can recover the image resource. The function below returns the image URL associated with a given page.

In [129]:
def get_image_urls(dataset_id: str, record_id: str) -> list[str]:
    """ Retrieves image URLs from the Presentation API
    """
    url = (
        f"https://iiif.europeana.eu/presentation/"
        f"{dataset_id}/{record_id}/manifest"
    )

    manifest = httpx.get(url, timeout=30.0).json()
    images = []
    sequences = manifest.get("sequences", [])

    for seq in sequences:
        for canvas in seq.get("canvases", []):

            for annotation in canvas.get("images", []):

                resource = annotation.get("resource", {})

                if isinstance(resource, dict):
                    image_id = resource.get("@id")

                    if image_id:
                        images.append(image_id)

    return images

# Use example from previous step
example = records_df.iloc[0]

imgs = get_image_urls(
    example.dataset_id,
    example.record_id,
)

print(f"Number of images:{len(imgs)}")
print(f"Links to images: {imgs[:5]}")

Number of images :80
Links to images : ['https://iiif.europeana.eu/image/WVI7PDW6LKS6JEEJ5RRCIKHVYQY7JCLV5DZ7D2DTFPRSJ6N6JO6Q/presentation_images/673f3ec0-020c-11e6-a696-fa163e2dd531/node-3/image/BNL/Journal_historique_et_littéraire/1792/06/15/00241/full/full/0/default.jpg', 'https://iiif.europeana.eu/image/WVI7PDW6LKS6JEEJ5RRCIKHVYQY7JCLV5DZ7D2DTFPRSJ6N6JO6Q/presentation_images/673f3ec0-020c-11e6-a696-fa163e2dd531/node-3/image/BNL/Journal_historique_et_littéraire/1792/06/15/00242/full/full/0/default.jpg', 'https://iiif.europeana.eu/image/WVI7PDW6LKS6JEEJ5RRCIKHVYQY7JCLV5DZ7D2DTFPRSJ6N6JO6Q/presentation_images/673f3ec0-020c-11e6-a696-fa163e2dd531/node-3/image/BNL/Journal_historique_et_littéraire/1792/06/15/00243/full/full/0/default.jpg', 'https://iiif.europeana.eu/image/WVI7PDW6LKS6JEEJ5RRCIKHVYQY7JCLV5DZ7D2DTFPRSJ6N6JO6Q/presentation_images/673f3ec0-020c-11e6-a696-fa163e2dd531/node-3/image/BNL/Journal_historique_et_littéraire/1792/06/15/00244/full/full/0/default.jpg', 'https://iiif.eu

## Step 3.2 : Extract text from the Text Content API

Now that we have `page_ids`, we can retrieve the OCR / transcription text for each page using the Europeana Text Content API.

Each `pageId` corresponds to a text layer extracted from the scanned page. Let's look at the first 500 characters of some pages from the Journal historique et littéraire - 1791-12-15.
> Note : you must specify the language again to make sure that you're pointing to the right resource.

In [130]:
example = records_df.iloc[0]

# Retrieve full ocr text for each pageID
for i, pid in enumerate(page_ids[:10]):

    url = (
        f"https://api.europeana.eu/fulltext/"
        f"{example.dataset_id}/{example.record_id}/{pid}"
    )

    r = httpx.get(url, params={"lang": "fr"})

    # Expose index, id, and text preview for our example
    print("\n====================")
    print("PAGE INDEX:", i)
    print("PAGE ID:", pid)

    data = r.json()

    text = data.get("value")

    if text:
        print("TEXT PREVIEW:")
        print(f"Extract: {text[:500]}...")
    else:
        print("TEXT: NONE")


PAGE INDEX: 0
PAGE ID: 02467f5
TEXT PREVIEW:
Extract: Journal hift. & lin.
* * * ~ 1 "
issatv
Lettre de D. P. D. P. à M. à l’auteur
du Journal, touchant un ouvrage attribué  à üngene.
M * * *. le 23 Avril 1793.
"V/oici nli doute touchant un article de Pan* * cicnn; bibliographie que je préfente à la dé* cifion des favans par la voie de votre Journal. C« doute regarde le Commentaire d’Origene fur VEpitre aux Romains. Ce Commentaire a été jufqu’à pré-fent attribué à Origene par tous ceux que j’ai pu eonuilter fur ce fujet , par Bellarmin , Annat, Wil...

PAGE INDEX: 1
PAGE ID: 03d9db1
TEXT PREVIEW:
Extract: Journal hifl. ô> lût,
i 4S
Mes chers concitoyens , je les obferve tous. Un dégoût général des plus faines maximes; Une invincible horreur des plus limpides eaux; La fureur de courir aux plus fales ruifleaux ; Une implacable foif & de fang & de crimes; Le trouble de l’efprit, le défordre des fens ; Des yeux en feu , d’effroyables aecens, Un levain corrofif, qui fourdement fermente Dan

## Step 3.3 : But how can we match text to images ?

Let's build a unified dataframe to preview image IIIF URLs, the associated OCR text, and some useful metadata.

#### Full use case example

This script demonstrates how to:

* load a Europeana IIIF manifest
* extract page identifiers from annotation links
* use those IDs to retrieve OCR text via the fulltext API
* build a structured list linking page → annotation → image → text

This is a minimal teaching example using a single record.

In [112]:
# For teaching only we take just one example 
example = records_df.iloc[0]
 
# Retrieve the IIIF manifest
manifest_url = (

    f"https://iiif.europeana.eu/presentation/"

    f"{example.dataset_id}/{example.record_id}/manifest"

)
 
response = httpx.get(manifest_url, timeout=30.0)
response.raise_for_status()
manifest = response.json()

In [ ]:
# Keep only canvases
canvases = manifest["sequences"][0]["canvases"]
 
def extract_page_id(other_content_url: str):

    """Retrieve pageID associated to each canvas using a regular expression.
    """

    try:

        match = re.search(r"/annopage/([^?]+)", other_content_url[0])
        
        if match:
            
            page_id = match.group(1)
            
            return page_id

    except:
        
        return None
        
    return None
 

In [ ]:
def extract_ocr_text(dataset_id: str, record_id: str, page_id: str, lang:str = "fr"):

    """Retrieve OCR full text content.
    """

    #print(dataset_id, record_id, page_id)

    text_content_url = (

    f"https://api.europeana.eu/fulltext/{dataset_id}/{record_id}/{page_id}"

)

    response = httpx.get(text_content_url, timeout=30.0)

    value = response.json()["value"]

    #response.raise_for_status()

    return value
 

In [131]:
# Retrieve pageID, otherContent URL, image URL and OCR text in tuple e.g. (6716, https://iiif.europeana.eu...?lang=fr, https://iiif.europeana.eu/image....jpg, Journal du lundi)

page_with_text_pairs = [(extract_page_id(canvas.get("otherContent")), canvas.get("otherContent")[0], canvas["images"][0]["resource"]["@id"]) for canvas in canvases if canvas.get("otherContent") is not None]
 
# Iterate over text_pairs to retrieve OCR text

text_pairs = [

    (page[0], page[1], page[2], 

    extract_ocr_text(

        dataset_id=example.dataset_id,

        record_id=example.record_id,

        page_id=page[0]))

for page in page_with_text_pairs]

In [ ]:
# Display the results inside a dataframe
df = pd.DataFrame(text_pairs, columns=["page_id","other_content_url","image_iiif_url","ocr_text"])
df.head()

### Saving file locally

You may want to save your results in a local file to further experiment with NLP options. 
This script shows you how to save your output as a CSV file :

In [ ]:
# Save the final dataset to a CSV file

output_path = "../output/journal_historique.csv" 

df_final.to_csv(output_path, index=False, encoding="utf-8")

print(f"File successfully saved to: {output_path}")